<a href="https://colab.research.google.com/github/akul-bharadwaj/various-agents/blob/main/Case_Study_6_LangChain_ReAct_Financial_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study 6: Build a ReAct Agent
## Financial Assistant — Reason, Act, Observe

### Objective

Build a financial assistant that:

- decides what data to fetch;
- calls tools dynamically;
- observes tool results;
- repeats the process when more evidence is needed; and
- produces a final educational recommendation.

The loop follows:

```text
Thought → Action → Observation → Thought → ...
```

In this notebook, **Thought** is a concise, user-visible decision summary. The notebook does not request or expose hidden chain-of-thought.

### Stack

- LangChain OpenAI
- LangChain tools
- Pydantic structured output
- Explicit Python ReAct loop

> **Disclaimer:** All profiles, goals, markets, and recommendations are fictional and educational. This is not personalised financial, investment, tax, or legal advice.

## ReAct workflow

```text
User request
    │
    ▼
Thought summary
    │
    ▼
Choose one action
    │
    ├── Financial profile tool
    ├── Risk profile tool
    ├── Goal constraints tool
    ├── Market snapshot tool
    └── FINAL
    │
    ▼
Observation
    │
    ├── More evidence needed ──► loop
    └── Enough evidence ───────► final recommendation
```

Restricting the model to one action per iteration makes the trace easy to inspect and helps demonstrate ReAct's limitation on long multi-step planning.

## 1. Install dependencies

In [1]:
!pip -q install --upgrade langchain langchain-openai pydantic pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 93.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.


## 2. Configure the OpenAI API key

In [2]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured.")

Enter your OpenAI API key: ··········
OpenAI API key configured.


## 3. Imports and configuration

In [3]:
import copy
import json
import logging
from importlib.metadata import version
from pprint import pprint
from typing import Any, Dict, List, Literal, Optional, Set

import pandas as pd
from langchain_core.tools import BaseTool, tool
from langchain_openai import ChatOpenAI
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    ValidationError,
    field_validator,
    model_validator,
)

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
)
logger = logging.getLogger("react_financial_agent")

print("Model:", MODEL)
print("langchain:", version("langchain"))
print("langchain-openai:", version("langchain-openai"))
print("pydantic:", version("pydantic"))

Model: gpt-4o-mini
langchain: 1.3.13
langchain-openai: 1.3.5
pydantic: 2.13.4


## 4. Fictional financial data

In [4]:
MOCK_FINANCIAL_PROFILES: Dict[str, Dict[str, Any]] = {
    "USR-1001": {
        "monthly_income": 120000.0,
        "monthly_expenses": 68000.0,
        "liquid_savings": 420000.0,
        "outstanding_debt": 180000.0,
        "monthly_debt_payment": 12000.0,
        "currency": "INR",
    },
    "USR-1002": {
        "monthly_income": 85000.0,
        "monthly_expenses": 62000.0,
        "liquid_savings": 90000.0,
        "outstanding_debt": 540000.0,
        "monthly_debt_payment": 18000.0,
        "currency": "INR",
    },
}

MOCK_RISK_PROFILES: Dict[str, Dict[str, Any]] = {
    "USR-1001": {
        "risk_tolerance": "moderate",
        "investment_horizon_years": 8,
        "liquidity_need": "medium",
        "loss_capacity": "moderate",
    },
    "USR-1002": {
        "risk_tolerance": "conservative",
        "investment_horizon_years": 3,
        "liquidity_need": "high",
        "loss_capacity": "low",
    },
}

MOCK_GOALS: Dict[str, Dict[str, Any]] = {
    "USR-1001": {
        "primary_goal": "house down payment",
        "target_amount": 2500000.0,
        "target_years": 7,
        "priority": "high",
        "currency": "INR",
    },
    "USR-1002": {
        "primary_goal": "emergency fund and debt reduction",
        "target_amount": 500000.0,
        "target_years": 2,
        "priority": "high",
        "currency": "INR",
    },
}

MOCK_MARKETS: Dict[str, Dict[str, Any]] = {
    "equity": {
        "asset_class": "equity",
        "illustrative_return_range_percent": "8–12",
        "volatility": "high",
        "liquidity": "high",
        "capital_preservation": "low",
        "outlook": (
            "Long-term growth potential with substantial "
            "short-term variability."
        ),
    },
    "bonds": {
        "asset_class": "bonds",
        "illustrative_return_range_percent": "5–8",
        "volatility": "low-to-medium",
        "liquidity": "medium-to-high",
        "capital_preservation": "medium",
        "outlook": (
            "Potential income and stability, subject to "
            "interest-rate and credit risk."
        ),
    },
    "gold": {
        "asset_class": "gold",
        "illustrative_return_range_percent": "variable",
        "volatility": "medium-to-high",
        "liquidity": "high",
        "capital_preservation": "medium",
        "outlook": (
            "Possible diversification benefit, but no "
            "assured income or return."
        ),
    },
    "cash": {
        "asset_class": "cash",
        "illustrative_return_range_percent": "3–7",
        "volatility": "low",
        "liquidity": "very high",
        "capital_preservation": "high",
        "outlook": (
            "Useful for short-term liquidity, though "
            "inflation can reduce purchasing power."
        ),
    },
}

print("Fictional datasets created.")

Fictional datasets created.


## 5. Tool input schemas

In [5]:
USER_ID_PATTERN = r"^USR-[0-9]{4}$"


class StrictSchema(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
    )


class UserLookupInput(StrictSchema):
    user_id: str = Field(
        pattern=USER_ID_PATTERN,
        description="Fictional user ID such as USR-1001.",
    )

    @field_validator("user_id", mode="before")
    @classmethod
    def normalise_user_id(cls, value: Any) -> Any:
        if isinstance(value, str):
            return value.strip().upper()
        return value


class MarketInput(StrictSchema):
    asset_class: Literal[
        "equity",
        "bonds",
        "gold",
        "cash",
    ]


print("Tool schemas defined.")

Tool schemas defined.


## 6. Implement four LangChain tools

In [6]:
class FictionalUserNotFoundError(Exception):
    pass


@tool(args_schema=UserLookupInput)
def get_financial_profile(user_id: str) -> str:
    """Get mock income, expenses, savings, and debt."""
    user_id = user_id.strip().upper()

    if user_id not in MOCK_FINANCIAL_PROFILES:
        raise FictionalUserNotFoundError(
            f"No fictional financial profile for {user_id}."
        )

    profile = copy.deepcopy(
        MOCK_FINANCIAL_PROFILES[user_id]
    )

    monthly_surplus = round(
        profile["monthly_income"]
        - profile["monthly_expenses"]
        - profile["monthly_debt_payment"],
        2,
    )

    emergency_months = round(
        profile["liquid_savings"]
        / max(profile["monthly_expenses"], 1.0),
        2,
    )

    result = {
        "status": "success",
        "evidence_key": "financial_profile",
        "user_id": user_id,
        **profile,
        "monthly_surplus_after_debt": monthly_surplus,
        "estimated_emergency_fund_months": emergency_months,
        "source": "fictional_financial_database",
    }

    return json.dumps(result, indent=2)


@tool(args_schema=UserLookupInput)
def get_risk_profile(user_id: str) -> str:
    """Get a mock risk tolerance and horizon profile."""
    user_id = user_id.strip().upper()

    if user_id not in MOCK_RISK_PROFILES:
        raise FictionalUserNotFoundError(
            f"No fictional risk profile for {user_id}."
        )

    result = {
        "status": "success",
        "evidence_key": "risk_profile",
        "user_id": user_id,
        **copy.deepcopy(MOCK_RISK_PROFILES[user_id]),
        "source": "fictional_risk_questionnaire",
    }

    return json.dumps(result, indent=2)


@tool(args_schema=UserLookupInput)
def get_goal_constraints(user_id: str) -> str:
    """Get a mock financial goal and deadline."""
    user_id = user_id.strip().upper()

    if user_id not in MOCK_GOALS:
        raise FictionalUserNotFoundError(
            f"No fictional goal record for {user_id}."
        )

    result = {
        "status": "success",
        "evidence_key": "goal_constraints",
        "user_id": user_id,
        **copy.deepcopy(MOCK_GOALS[user_id]),
        "source": "fictional_goal_database",
    }

    return json.dumps(result, indent=2)


@tool(args_schema=MarketInput)
def get_market_snapshot(asset_class: str) -> str:
    """Get a mock educational snapshot for one asset class."""
    asset_class = asset_class.strip().lower()

    result = {
        "status": "success",
        "evidence_key": f"market_snapshot:{asset_class}",
        **copy.deepcopy(MOCK_MARKETS[asset_class]),
        "source": "fictional_market_service",
        "disclaimer": (
            "Illustrative mock assumptions, not a forecast."
        ),
    }

    return json.dumps(result, indent=2)


TOOLS: List[BaseTool] = [
    get_financial_profile,
    get_risk_profile,
    get_goal_constraints,
    get_market_snapshot,
]

TOOL_REGISTRY: Dict[str, BaseTool] = {
    item.name: item for item in TOOLS
}

print("Available tools:", list(TOOL_REGISTRY))

Available tools: ['get_financial_profile', 'get_risk_profile', 'get_goal_constraints', 'get_market_snapshot']


## 7. Structured ReAct decision

The model returns one concise thought summary and one action per iteration.

In [7]:
ActionName = Literal[
    "get_financial_profile",
    "get_risk_profile",
    "get_goal_constraints",
    "get_market_snapshot",
    "FINAL",
]


class ReActDecision(BaseModel):
    model_config = ConfigDict(extra="forbid")

    thought_summary: str = Field(
        min_length=5,
        max_length=300,
        description=(
            "Brief user-visible reason for the next action. "
            "Do not provide hidden chain-of-thought."
        ),
    )
    action: ActionName
    user_id: Optional[str] = None
    asset_class: Optional[
        Literal["equity", "bonds", "gold", "cash"]
    ] = None
    final_recommendation: Optional[str] = Field(
        default=None,
        max_length=1800,
    )

    @model_validator(mode="after")
    def validate_fields(self) -> "ReActDecision":
        user_tools = {
            "get_financial_profile",
            "get_risk_profile",
            "get_goal_constraints",
        }

        if self.action in user_tools:
            if not self.user_id:
                raise ValueError(
                    "user_id is required for this tool."
                )
            if self.asset_class is not None:
                raise ValueError(
                    "asset_class is not valid for this tool."
                )
            if self.final_recommendation is not None:
                raise ValueError(
                    "Tool actions cannot contain a final answer."
                )

        elif self.action == "get_market_snapshot":
            if self.asset_class is None:
                raise ValueError(
                    "asset_class is required for market data."
                )
            if self.user_id is not None:
                raise ValueError(
                    "user_id is not valid for market data."
                )
            if self.final_recommendation is not None:
                raise ValueError(
                    "Tool actions cannot contain a final answer."
                )

        elif self.action == "FINAL":
            if not self.final_recommendation:
                raise ValueError(
                    "FINAL requires a recommendation."
                )
            if (
                self.user_id is not None
                or self.asset_class is not None
            ):
                raise ValueError(
                    "FINAL cannot contain tool arguments."
                )

        return self


print("ReActDecision schema defined.")

ReActDecision schema defined.


## 8. Configure the structured decision model

In [8]:
react_llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
)

react_model = react_llm.with_structured_output(
    ReActDecision,
    method="json_schema",
)

SYSTEM_INSTRUCTIONS = """
You control a fictional educational financial assistant.

At every iteration:
1. Write one concise public thought_summary.
2. Select exactly one tool action or FINAL.
3. Use only supplied observations.
4. Never repeat an identical tool call.
5. Never invent missing financial or market information.
6. Do not choose FINAL while required_evidence is missing.
7. The final answer must cover the current position,
   goal/risk implications, educational next steps,
   uncertainty, and limitations.
8. Never promise returns or make a regulated suitability decision.
9. State that all data is fictional and professional advice
   may be needed.
10. Do not reveal hidden chain-of-thought.
"""

print("Structured ReAct model configured.")

Structured ReAct model configured.


## 9. Safe action execution

In [9]:
def action_arguments(
    decision: ReActDecision,
) -> Dict[str, Any]:
    if decision.action in {
        "get_financial_profile",
        "get_risk_profile",
        "get_goal_constraints",
    }:
        return {"user_id": decision.user_id}

    if decision.action == "get_market_snapshot":
        return {"asset_class": decision.asset_class}

    return {}


def safe_execute_action(
    decision: ReActDecision,
) -> Dict[str, Any]:
    if decision.action == "FINAL":
        raise ValueError("FINAL is not a tool action.")

    try:
        tool_object = TOOL_REGISTRY[decision.action]
        raw_output = tool_object.invoke(
            action_arguments(decision)
        )
        output = json.loads(raw_output)

        if not isinstance(output, dict):
            raise ValueError(
                "Tool output must be a JSON object."
            )

        return output

    except ValidationError as exc:
        return {
            "status": "error",
            "error_code": "INVALID_TOOL_INPUT",
            "message": str(exc),
            "retryable": False,
        }

    except FictionalUserNotFoundError as exc:
        return {
            "status": "error",
            "error_code": "USER_NOT_FOUND",
            "message": str(exc),
            "retryable": False,
        }

    except Exception as exc:
        return {
            "status": "error",
            "error_code": "TOOL_FAILURE",
            "message": f"{type(exc).__name__}: {exc}",
            "retryable": False,
        }


print("Safe executor defined.")

Safe executor defined.


## 10. Implement the explicit ReAct loop

In [10]:
class ReActFinancialAgent:
    """Manual Thought → Action → Observation loop."""

    def __init__(
        self,
        max_steps: int = 6,
        max_model_errors: int = 2,
    ):
        self.max_steps = max_steps
        self.max_model_errors = max_model_errors

    def run(
        self,
        user_request: str,
        required_evidence: Optional[Set[str]] = None,
        verbose: bool = True,
    ) -> Dict[str, Any]:
        required_evidence = set(
            required_evidence or set()
        )

        observations: List[Dict[str, Any]] = []
        trace: List[Dict[str, Any]] = []
        collected_evidence: Set[str] = set()
        executed_signatures: Set[str] = set()
        model_errors = 0

        for step in range(1, self.max_steps + 1):
            missing_evidence = sorted(
                required_evidence - collected_evidence
            )

            context = {
                "user_request": user_request,
                "required_evidence": sorted(
                    required_evidence
                ),
                "collected_evidence": sorted(
                    collected_evidence
                ),
                "missing_evidence": missing_evidence,
                "observations": observations,
                "executed_actions": sorted(
                    executed_signatures
                ),
                "remaining_action_budget": (
                    self.max_steps - step + 1
                ),
            }

            prompt = (
                SYSTEM_INSTRUCTIONS
                + "\n\nCurrent execution context:\n"
                + json.dumps(
                    context,
                    indent=2,
                    default=str,
                )
            )

            try:
                decision = react_model.invoke(prompt)
                model_errors = 0
            except Exception as exc:
                model_errors += 1
                event = {
                    "step": step,
                    "thought_summary": (
                        "The model did not return a validated "
                        "decision."
                    ),
                    "action": "MODEL_ERROR",
                    "action_input": {},
                    "observation": {
                        "status": "error",
                        "error_code": (
                            "MODEL_DECISION_FAILURE"
                        ),
                        "message": (
                            f"{type(exc).__name__}: {exc}"
                        ),
                    },
                }
                trace.append(event)

                if verbose:
                    self._print_event(event)

                if model_errors >= self.max_model_errors:
                    return self._result(
                        user_request,
                        trace,
                        collected_evidence,
                        required_evidence,
                        None,
                        "MAX_MODEL_ERRORS_REACHED",
                    )
                continue

            args = action_arguments(decision)

            if decision.action == "FINAL":
                missing_evidence = sorted(
                    required_evidence - collected_evidence
                )

                if missing_evidence:
                    observation = {
                        "status": "error",
                        "error_code": (
                            "PREMATURE_FINAL_ANSWER"
                        ),
                        "message": (
                            "Required evidence is still missing."
                        ),
                        "missing_evidence": missing_evidence,
                    }
                    event = {
                        "step": step,
                        "thought_summary": (
                            decision.thought_summary
                        ),
                        "action": "FINAL",
                        "action_input": {},
                        "observation": observation,
                    }
                    trace.append(event)

                    if verbose:
                        self._print_event(event)
                    continue

                event = {
                    "step": step,
                    "thought_summary": (
                        decision.thought_summary
                    ),
                    "action": "FINAL",
                    "action_input": {},
                    "observation": {
                        "status": "completed"
                    },
                }
                trace.append(event)

                if verbose:
                    self._print_event(event)

                return self._result(
                    user_request,
                    trace,
                    collected_evidence,
                    required_evidence,
                    decision.final_recommendation,
                    "MODEL_RETURNED_FINAL",
                )

            signature = json.dumps(
                {
                    "action": decision.action,
                    "arguments": args,
                },
                sort_keys=True,
            )

            if signature in executed_signatures:
                observation = {
                    "status": "error",
                    "error_code": (
                        "DUPLICATE_ACTION_PREVENTED"
                    ),
                    "message": (
                        "The same action and arguments "
                        "were already used."
                    ),
                }
            else:
                executed_signatures.add(signature)
                observation = safe_execute_action(
                    decision
                )

                evidence_key = observation.get(
                    "evidence_key"
                )
                if (
                    observation.get("status") == "success"
                    and evidence_key
                ):
                    collected_evidence.add(
                        evidence_key
                    )

                observations.append(
                    {
                        "action": decision.action,
                        "action_input": args,
                        "result": observation,
                    }
                )

            event = {
                "step": step,
                "thought_summary": (
                    decision.thought_summary
                ),
                "action": decision.action,
                "action_input": args,
                "observation": observation,
            }
            trace.append(event)

            if verbose:
                self._print_event(event)

        return self._result(
            user_request,
            trace,
            collected_evidence,
            required_evidence,
            None,
            "MAX_STEPS_REACHED",
        )

    @staticmethod
    def _result(
        user_request: str,
        trace: List[Dict[str, Any]],
        collected_evidence: Set[str],
        required_evidence: Set[str],
        final_recommendation: Optional[str],
        termination_reason: str,
    ) -> Dict[str, Any]:
        missing = sorted(
            required_evidence - collected_evidence
        )

        coverage = (
            round(
                len(
                    required_evidence
                    & collected_evidence
                )
                / len(required_evidence),
                4,
            )
            if required_evidence
            else 1.0
        )

        return {
            "user_request": user_request,
            "completed": (
                final_recommendation is not None
                and not missing
            ),
            "final_recommendation": (
                final_recommendation
            ),
            "termination_reason": termination_reason,
            "steps_used": len(trace),
            "required_evidence": sorted(
                required_evidence
            ),
            "collected_evidence": sorted(
                collected_evidence
            ),
            "missing_evidence": missing,
            "evidence_coverage": coverage,
            "trace": trace,
        }

    @staticmethod
    def _print_event(
        event: Dict[str, Any],
    ) -> None:
        print(
            f"\n{'=' * 18} "
            f"Step {event['step']} "
            f"{'=' * 18}"
        )
        print(
            "Thought summary:",
            event["thought_summary"],
        )
        print("Action:", event["action"])
        print("Action input:")
        pprint(event["action_input"])
        print("Observation:")
        pprint(event["observation"])


print("ReActFinancialAgent defined.")

ReActFinancialAgent defined.


## 11. Successful focused task

This task requires three evidence items. A six-step budget permits three tool calls and a final response.

In [11]:
focused_evidence = {
    "financial_profile",
    "risk_profile",
    "goal_constraints",
}

focused_agent = ReActFinancialAgent(
    max_steps=6
)

focused_result = focused_agent.run(
    user_request=(
        "For fictional user USR-1001, provide an "
        "educational high-level recommendation for "
        "progressing towards the recorded financial "
        "goal. Consider cash flow, risk profile, and "
        "goal constraints. Do not name securities or "
        "promise returns."
    ),
    required_evidence=focused_evidence,
)

print("\nFocused-task status")
pprint(
    {
        "completed": focused_result["completed"],
        "termination_reason": focused_result[
            "termination_reason"
        ],
        "steps_used": focused_result["steps_used"],
        "evidence_coverage": focused_result[
            "evidence_coverage"
        ],
        "missing_evidence": focused_result[
            "missing_evidence"
        ],
    }
)

print("\nFinal recommendation:")
print(focused_result["final_recommendation"])


================== Step 1 ==================
Thought summary: I need to gather the user's financial profile, risk profile, and goal constraints to provide a tailored recommendation.
Action: get_financial_profile
Action input:
{'user_id': 'USR-1001'}
Observation:
{'currency': 'INR',
 'estimated_emergency_fund_months': 6.18,
 'evidence_key': 'financial_profile',
 'liquid_savings': 420000.0,
 'monthly_debt_payment': 12000.0,
 'monthly_expenses': 68000.0,
 'monthly_income': 120000.0,
 'monthly_surplus_after_debt': 40000.0,
 'outstanding_debt': 180000.0,
 'source': 'fictional_financial_database',
 'status': 'success',
 'user_id': 'USR-1001'}

================== Step 2 ==================
Thought summary: I need to gather the missing goal constraints and risk profile to provide a comprehensive recommendation.
Action: get_goal_constraints
Action input:
{'user_id': 'USR-1001'}
Observation:
{'currency': 'INR',
 'evidence_key': 'goal_constraints',
 'primary_goal': 'house down payment',
 'priorit

## 12. Display the Thought → Action → Observation log

In [12]:
def trace_dataframe(
    trace: List[Dict[str, Any]],
) -> pd.DataFrame:
    rows = []

    for event in trace:
        observation = event.get("observation", {})
        rows.append(
            {
                "Step": event["step"],
                "Thought summary": event[
                    "thought_summary"
                ],
                "Action": event["action"],
                "Action input": event[
                    "action_input"
                ],
                "Observation status": (
                    observation.get("status")
                    if isinstance(observation, dict)
                    else None
                ),
                "Evidence key": (
                    observation.get("evidence_key")
                    if isinstance(observation, dict)
                    else None
                ),
                "Error code": (
                    observation.get("error_code")
                    if isinstance(observation, dict)
                    else None
                ),
            }
        )

    return pd.DataFrame(rows)


display(trace_dataframe(focused_result["trace"]))

,Step,Thought summary,Action,Action input,Observation status,Evidence key,Error code
0,1,"I need to gather the user's financial profile,...",get_financial_profile,{'user_id': 'USR-1001'},success,financial_profile,NaN
1,2,I need to gather the missing goal constraints ...,get_goal_constraints,{'user_id': 'USR-1001'},success,goal_constraints,NaN
2,3,"To provide a comprehensive recommendation, I n...",get_risk_profile,{'user_id': 'USR-1001'},success,risk_profile,NaN
3,4,The model did not return a validated decision.,MODEL_ERROR,{},error,NaN,MODEL_DECISION_FAILURE
4,5,The model did not return a validated decision.,MODEL_ERROR,{},error,NaN,MODEL_DECISION_FAILURE


# 13. Limitation: long multi-step planning

The complex task requires six evidence items:

1. financial profile;
2. risk profile;
3. goal constraints;
4. equity snapshot;
5. bond snapshot;
6. gold snapshot.

The ReAct controller can perform only **one action per iteration**, but receives a budget of only **four steps**.

Even an ideal reactive policy cannot collect six evidence items and produce a final recommendation within four single-action iterations.

In [13]:
complex_evidence = {
    "financial_profile",
    "risk_profile",
    "goal_constraints",
    "market_snapshot:equity",
    "market_snapshot:bonds",
    "market_snapshot:gold",
}

limited_agent = ReActFinancialAgent(
    max_steps=4
)

limited_result = limited_agent.run(
    user_request=(
        "For fictional user USR-1001, build a detailed "
        "multi-stage roadmap. Analyse cash flow, risk "
        "capacity, goal constraints, and compare equity, "
        "bonds, and gold before making a recommendation. "
        "Do not guess missing information."
    ),
    required_evidence=complex_evidence,
)

print("\nLong-task status")
pprint(
    {
        "completed": limited_result["completed"],
        "termination_reason": limited_result[
            "termination_reason"
        ],
        "steps_used": limited_result["steps_used"],
        "evidence_coverage": limited_result[
            "evidence_coverage"
        ],
        "collected_evidence": limited_result[
            "collected_evidence"
        ],
        "missing_evidence": limited_result[
            "missing_evidence"
        ],
    }
)

print("\nFinal recommendation:")
print(limited_result["final_recommendation"])


================== Step 1 ==================
Thought summary: To build a detailed roadmap, I need to gather the user's financial profile and risk profile first.
Action: get_financial_profile
Action input:
{'user_id': 'USR-1001'}
Observation:
{'currency': 'INR',
 'estimated_emergency_fund_months': 6.18,
 'evidence_key': 'financial_profile',
 'liquid_savings': 420000.0,
 'monthly_debt_payment': 12000.0,
 'monthly_expenses': 68000.0,
 'monthly_income': 120000.0,
 'monthly_surplus_after_debt': 40000.0,
 'outstanding_debt': 180000.0,
 'source': 'fictional_financial_database',
 'status': 'success',
 'user_id': 'USR-1001'}

================== Step 2 ==================
Thought summary: I need to gather the user's goal constraints to understand their financial objectives better.
Action: get_goal_constraints
Action input:
{'user_id': 'USR-1001'}
Observation:
{'currency': 'INR',
 'evidence_key': 'goal_constraints',
 'primary_goal': 'house down payment',
 'priority': 'high',
 'source': 'fictional

## 14. Inspect the failed long-task trace

In [14]:
display(trace_dataframe(limited_result["trace"]))

,Step,Thought summary,Action,Action input,Observation status,Evidence key,Error code
0,1,"To build a detailed roadmap, I need to gather ...",get_financial_profile,{'user_id': 'USR-1001'},success,financial_profile,NaN
1,2,I need to gather the user's goal constraints t...,get_goal_constraints,{'user_id': 'USR-1001'},success,goal_constraints,NaN
2,3,I need to gather the risk profile and market s...,get_risk_profile,{'user_id': 'USR-1001'},success,risk_profile,NaN
3,4,The model did not return a validated decision.,MODEL_ERROR,{},error,NaN,MODEL_DECISION_FAILURE


## 15. Compare the outputs

In [15]:
comparison = pd.DataFrame(
    [
        {
            "Run": "Focused task",
            "Required evidence": len(
                focused_result["required_evidence"]
            ),
            "Maximum steps": focused_agent.max_steps,
            "Steps used": focused_result["steps_used"],
            "Evidence coverage": focused_result[
                "evidence_coverage"
            ],
            "Completed": focused_result["completed"],
            "Termination": focused_result[
                "termination_reason"
            ],
        },
        {
            "Run": "Long multi-step task",
            "Required evidence": len(
                limited_result["required_evidence"]
            ),
            "Maximum steps": limited_agent.max_steps,
            "Steps used": limited_result["steps_used"],
            "Evidence coverage": limited_result[
                "evidence_coverage"
            ],
            "Completed": limited_result["completed"],
            "Termination": limited_result[
                "termination_reason"
            ],
        },
    ]
)

display(comparison)

,Run,Required evidence,Maximum steps,Steps used,Evidence coverage,Completed,Termination
0,Focused task,3,6,5,1.0,False,MAX_MODEL_ERRORS_REACHED
1,Long multi-step task,6,4,4,0.5,False,MAX_STEPS_REACHED


## Why ReAct can fail on long planning tasks

ReAct selects the next action from the current context. It does not necessarily construct and reserve resources for a complete global plan before execution.

Common limitations:

- **short-horizon action selection** — a locally sensible step may not support the best overall sequence;
- **step-budget exhaustion** — dependent tool calls consume the available loop budget;
- **premature completion attempts** — the model may answer before gathering all evidence;
- **redundant calls** — an unguarded agent may repeat an action;
- **growing context** — long observation histories can distract later decisions;
- **weak dependency representation** — prerequisites are not always explicitly modelled.

For larger tasks, a planner–executor or graph workflow can first create a task list, represent dependencies, and then execute each item systematically.

## 16. Optional larger-budget experiment

Giving the same ReAct agent more steps may allow completion, but more steps alone do not guarantee efficient or globally optimal planning.

In [ ]:
# Uncomment to experiment.
#
# expanded_agent = ReActFinancialAgent(
#     max_steps=10
# )
#
# expanded_result = expanded_agent.run(
#     user_request=(
#         "For fictional user USR-1001, analyse cash "
#         "flow, risk, goals, equity, bonds, and gold "
#         "before producing a roadmap."
#     ),
#     required_evidence=complex_evidence,
# )
#
# pprint(
#     {
#         "completed": expanded_result["completed"],
#         "steps_used": expanded_result["steps_used"],
#         "evidence_coverage": expanded_result[
#             "evidence_coverage"
#         ],
#         "missing_evidence": expanded_result[
#             "missing_evidence"
#         ],
#     }
# )

## Requirements mapping

| Requirement | Implementation |
|---|---|
| Thought → action → observation | `ReActFinancialAgent.run()` |
| Log reasoning steps | Concise public trace and DataFrame |
| At least two tools | Four LangChain tools |
| Decide what to fetch | Structured model decision per iteration |
| Call tools | `safe_execute_action()` |
| Produce recommendation | `FINAL` action |
| Handle failures | Error observation objects |
| Avoid duplicate calls | Executed-action signatures |
| Prevent endless execution | `max_steps` |
| Demonstrate long-planning limitation | Six evidence items with four-step budget |
| Compare results | Focused-versus-long-task table |

## Key learning

ReAct works well for short, adaptive tool workflows:

```text
brief reason → one action → inspect observation → continue
```

It becomes less reliable when a task needs a long, dependency-aware plan before execution. A planner–executor design is usually clearer for such tasks.

## References

- LangChain tools:  
  https://docs.langchain.com/oss/python/langchain/tools

- LangChain models and structured output:  
  https://docs.langchain.com/oss/python/langchain/models

- LangChain OpenAI integration:  
  https://docs.langchain.com/oss/python/integrations/chat/openai

- LangChain agents:  
  https://docs.langchain.com/oss/python/langchain/agents

- ReAct paper:  
  https://arxiv.org/abs/2210.03629